# Tutorial 3: Learn about `Selenium`

## Kailyn Lau

Follow along with this [RealPython tutorial on Selenium](https://realpython.com/modern-web-automation-with-python-and-selenium/).

**Table of Contents**

1. [Understand the Project and Approach](#sec1)
2. [Set Up Your Environment](#sec2)
3. [Navigate a Web Page With Python and Selenium](#sec3)
4. [Interact With Web Elements](#sec4)
5. [Handle Dynamic Content](#sec5)
6. [Implement the Page Object Model (POM)](#sec6)

<h2 id="sec1">1. Understand the Project and Approach</h2>


 - Requests/Beautiful Soup/Scrapy are great when everything is already available in the HTTP response, but Selenium is good when I need to make a little robot behave like me in a browser
 - Especially good when JavaScript creates or updates the content

### Page Object Model

The POM idea is to put the knowledge of *how the website works* into page/component classes. (like Svelte?)

<h2 id="sec2">2. Set Up Your Environment</h2>

I ignored the geckodriver part because I'm using Chrome, not firefox.

In [1]:
# pip install selenium

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


<h2 id="sec3">3. Navigate a Web Page With Python and Selenium</h2>

 - Remember to inspect the page manually!
 - And look out for any cookie consent forms or whatnot - selenium will also have to click through those.

The tutorial's simplified Bandcamp structure is roughly:

- a `results-grid` container
- `results-grid-item` elements for individual tracks
- a `play-pause-button`
- album/artist information inside `.meta`
- optional genre information

### Locating elements

Selenium provides several locator strategies, including By.ID, .CLASS_NAME, .CSS_SELECTOR, .XPATH, .TAG_NAME, .NAME (class), etc.
 - `find_element()` → one element
 - `find_elements()` → list of elements


An absolute XPath such as:

```python
# /html/body/div[2]/span[1]/a[3]
```

can work, but it is extremely fragile. If someone inserts one extra `<div>`, it doesn't work anymore. Intead, use stable IDs or semantic/stable classes when possible.

<h2 id='sec4'>4. Interact With Web Elements</h2>

In [ ]:
# button = driver.find_element(By.ID, "submit-button")
# button.click()

# search_box = driver.find_element(By.TAG_NAME, "input")
# search_box.send_keys("selenium") # types into an input
# search_box.submit()

# Or explicitly send Enter:
# from selenium.webdriver.common.keys import Keys
# search_box.send_keys(Keys.ENTER)

# cookie/overlay handling
# from selenium.common.exceptions import NoSuchElementException

# try:
#     cookie_button = driver.find_element(
#         By.CSS_SELECTOR,
#         "#cookie-control-dialog button.g-button.outline",
#     )
#     cookie_button.click()
# except NoSuchElementException:
#     pass

# Alternative for some situations:
# driver.execute_script(
#     "arguments[0].click();",
#     overlay_element,
# )
#
# The JavaScript approach can bypass normal interaction behavior,
# so it isn't necessarily appropriate for realistic testing.
# driver.execute_script("arguments[0].click();", overlay_element)

`ActionChains` can compose interactions such as hovering and clicking.



In [ ]:
# from selenium.webdriver import ActionChains

# menu = driver.find_element(By.CSS_SELECTOR, ".menu")
# submenu = driver.find_element(By.CSS_SELECTOR, ".menu #submenu")

# actions = ActionChains(driver)
# actions.move_to_element(menu)
# actions.click(submenu)
# actions.perform()


A form can be located first, then its individual inputs can be filled before submitting the form.



In [2]:
# signup = driver.find_element(By.ID, "signup-form")

# email = signup.find_element(By.NAME, "email")
# password = signup.find_element(By.NAME, "password")

# email.send_keys("user@example.com")
# password.send_keys("example-password")

# signup.submit()

<h2 id='sec5'>5. Handle Dynamic Content</h2>

It's better to use an implicit wait instead of a sleep().

Implicit Wait: Selenium polls the DOM for a specified time whenever you try to find an element.

Explicit Wait: Selenium applies conditions to waits, which makes it more flexible.

Fluent Wait: Selenium allows you to specify the polling interval, ignore certain exceptions that occur during polling, and set custom timeout messages.


In [ ]:
# driver.implicitly_wait(5)

### Explicit wait

Use `WebDriverWait` with an expected condition.

| Function | Description |
|-|-|
| presence_of_element_located() | Waits for an element to be present in the DOM. | 
| visibility_of_element_located() | Waits for an element to be present and visible. | 
| element_to_be_clickable() | Waits for an element to be visible and enabled for clicking. |
| alert_is_present() | Waits for a JavaScript alert to appear. |
| title_is() / title_contains() | Checks if the page title exactly matches or contains a substring. |
| url_to_be() / url_contains() | Verifies that the current URL exactly matches or contains a substring. |



In [ ]:
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC

# wait = WebDriverWait(driver, 10)

# # Wait without using the returned element:
# wait.until(
#     EC.element_to_be_clickable((By.ID, "view-more"))
# )

# # Or use the returned element directly:
# pagination = wait.until(
#     EC.element_to_be_clickable((By.ID, "view-more"))
# )
# pagination.click()


### Custom wait conditions

I can also pass my own function to `wait.until()`. This is useful when none of Selenium's built-in conditions express the exact state I care about.


In [ ]:
# wait = WebDriverWait(driver, 10)

# def tracks_loaded(driver):
#     cards = driver.find_elements(
#         By.CLASS_NAME,
#         "results-grid-item",
#     )
#     return any(card.text.strip() for card in cards)

# wait.until(tracks_loaded)


It's best to wait for the most stable indication that the page has finished its asynchronous work.

Headless mode can also behave differently from visible mode, so removing `--headless` is a useful debugging technique.

JavaScript alerts can be handled similarly:


In [ ]:
# from selenium.common.exceptions import NoAlertPresentException

# try:
#     alert = driver.switch_to.alert
#     alert.dismiss()

#     # Or:
#     # alert.accept()
# except NoAlertPresentException:
#     pass


<h2 id='sec6'>6. Implement the Page Object Model (POM)</h2>

### Project structure

The tutorial separates the browser-facing code into:

```text
bandcamp/
├── __init__.py
├── base.py
├── elements.py
├── locators.py
└── pages.py
```

Conceptually:

```text
DiscoverPage
    └── TrackListElement
            └── TrackElement
```

`locators.py` owns selectors, `elements.py` owns component behavior, and `pages.py` describes the page-level object.


### `base.py`

The base classes centralize common WebDriver setup, window sizing, and waiting behavior.


### `pages.py`

The page object represents only the parts of the Discover page that the application needs.

It handles cookie consent and exposes the track list.


### `elements.py`: track list

The track list component waits for track text, finds all track cards, filters out invisible/empty ones, and wraps them in `TrackElement` objects.


### `TrackElement`: user-level actions

The useful abstraction is that higher-level code can say `.play()` and `.pause()` instead of knowing how the button is located.



The track object also extracts metadata. The important ideas are:

- retrieve the album URL from `href`
- strip a query string if present
- tolerate tracks without a genre
- return a clean representation of album/artist/genre/URL

The tutorial initially uses a dictionary and later replaces it with a `dataclass`.


### `locators.py`

This is a particularly useful organizational choice because selectors are likely to change independently of application logic.

Each locator is stored as a `(strategy, value)` tuple, so `*locator` can be passed directly into Selenium.


### POM smoke test

Once the classes exist, the nice part is that the raw selectors disappear from the test/use site.


## Summary
### Selenium fundamentals

```text
WebDriver
  ↓
get(url)
  ↓
find_element / find_elements
  ↓
WebElement
  ↓
click / send_keys / submit / get_attribute / text
```

### Waiting

**Implicit wait:** broad safety net.

**Explicit wait:** wait for a specific condition.

**`time.sleep()`:** okay for quick demonstrations/debugging, but usually the least robust option.

### POM

```text
Application
    ↓
Page object
    ↓
Component object
    ↓
Locator
    ↓
Selenium WebDriver
    ↓
Actual browser
```

The application should not need to know that the play button happens to be a CSS selector today.